# 🏆 Reward Model Training — Llama-3.2-1B

> **ရည်ရွယ်ချက်**: Reward Model ဆိုတာဘာလဲ၊ ဘယ်လို train လဲ **အသေးစိတ်** နားလည်ဖို့  
> **Model**: `meta-llama/Llama-3.2-1B` (locally cached)  
> **GPU**: RTX 3050 12GB | **Approach**: Pure PyTorch + PEFT (LoRA)  
> **Dataset**: Sample preference data (hand-crafted for learning)

---

## 📋 Notebook Workflow

```
Step 1: Reward Model ဆိုတာ ဘာလဲ? — Theory & Intuition
Step 2: Setup — Dependencies & Model Loading
Step 3: Sample Preference Dataset — Human preference data ဘယ်လိုဖန်တီးလဲ
Step 4: Reward Model Architecture — CausalLM → Reward Model ဘယ်လိုပြောင်းလဲ
Step 5: Data Processing — Tokenization & DataLoader
Step 6: Training Loop — Bradley-Terry Loss နဲ့ train
Step 7: Evaluation — Model က ဘယ်လောက်ကောင်းလဲ
Step 8: Inference Demo — RM ကိုသုံးပြီး responses ကို score/rank လုပ်
```

## 📖 Step 1: Reward Model ဆိုတာ ဘာလဲ?

### 🤔 ဘာကြောင့် Reward Model လိုအပ်သလဲ?

LLM ကို RLHF (Reinforcement Learning from Human Feedback) နဲ့ align လုပ်ဖို့ **reward signal** လိုပါတယ်။  
Human annotator ကို real-time ခေါ်ပြီး score ပေးခိုင်းလို့ မဖြစ်နိုင်ဘူး။ ဒါကြောင့် **Reward Model** ကို train ပြီး human preferences ကို **approximate** လုပ်ပါတယ်။

### 🏗️ Reward Model Architecture

```
Input: [prompt + response]
    ↓
Transformer (LLM backbone — e.g., Llama-3.2-1B)
    ↓
Last Token's Hidden State (dim=2048)
    ↓
Linear Layer (2048 → 1)
    ↓
Output: Scalar Reward Score (e.g., 3.72)
```

### 📊 Training Data Format — Preference Pairs

Reward Model ကို **comparison data** (preference pairs) နဲ့ train ပါတယ်:

| Component | Description | Example |
|-----------|-------------|---------|
| **Prompt** (x) | User's question/instruction | "Python list ကို reverse ဘယ်လိုလုပ်မလဲ?" |
| **Chosen** (y_w) | Better response (preferred) | "list[::-1] ဒါမှမဟုတ် list.reverse() ကိုသုံးပါ" |
| **Rejected** (y_l) | Worse response | "Python ဆိုတာ programming language တစ်ခုပါ" |

### 📐 Bradley-Terry Model — Loss Function

Reward Model ကို train ရာမှာ **Bradley-Terry preference model** ကိုသုံးပါတယ်:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E} \left[ \log \sigma \left( r_\theta(x, y_w) - r_\theta(x, y_l) \right) \right]$$

**Intuition**: Chosen response ရဲ့ reward score က Rejected response ထက် **မြင့်ရမယ်**။ ဒီ gap ကို sigmoid ထဲထည့်ပြီး log probability ကို maximize လုပ်တာပါ။

```
r_θ(x, chosen) = 4.2   ─┐
                         ├─ gap = 4.2 - 1.3 = 2.9 → σ(2.9) ≈ 0.95 → -log(0.95) ≈ 0.05 (low loss ✅)
r_θ(x, rejected) = 1.3  ─┘

r_θ(x, chosen) = 2.1   ─┐
                         ├─ gap = 2.1 - 2.0 = 0.1 → σ(0.1) ≈ 0.52 → -log(0.52) ≈ 0.65 (high loss ❌)
r_θ(x, rejected) = 2.0  ─┘
```

In [ ]:
# ============================================================
# 📦 Step 2: Setup — Dependencies & Model Loading
# ============================================================

import subprocess, sys

packages = [
    "torch",
    "transformers>=4.43.0",
    "peft>=0.12.0",
    "accelerate",
]

for pkg in packages:
    try:
        pkg_name = pkg.split(">=")[0].split("==")[0]
        __import__(pkg_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages ready!")

In [ ]:
# ============================================================
# 📦 Step 2.1: Imports & Configuration
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# -----------------------------------------------------------
# Configuration
# -----------------------------------------------------------
MODEL_ID = "meta-llama/Llama-3.2-1B"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LENGTH = 256          # Token limit (reward model မှာ response တိုတိုပဲလိုတယ်)
BATCH_SIZE = 2            # RTX 3050 12GB အတွက် safe
LEARNING_RATE = 1e-4      # LoRA fine-tuning rate
NUM_EPOCHS = 3            # Sample data နည်းလို့ 3 epoch ပဲ
LORA_R = 16               # LoRA rank
LORA_ALPHA = 32           # LoRA alpha

print(f"🖥️  Device: {DEVICE}")
print(f"🧠 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"📊 VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# ============================================================
# 📦 Step 2.2: Load Base Model & Tokenizer
# ============================================================

print(f"📥 Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Llama tokenizer မှာ pad_token မရှိဘူး — eos_token ကိုသုံးမယ်
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"📥 Loading base model: {MODEL_ID}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"\n✅ Model loaded!")
print(f"📊 Parameters: {sum(p.numel() for p in base_model.parameters()) / 1e6:.1f}M")
print(f"📏 Hidden size: {base_model.config.hidden_size}")
print(f"📚 Vocab size: {base_model.config.vocab_size}")

## 📊 Step 3: Sample Preference Dataset

### Data Format ရှင်းပြချက်

Reward Model ကို train ဖို့ **preference pairs** လိုပါတယ်:
- **prompt**: User ရဲ့ question/instruction
- **chosen**: Human annotator က ပိုကောင်းတယ်လို့ ရွေးတဲ့ response  
- **rejected**: အရည်အသွေး နိမ့်တဲ့ response

### ဘယ်လို Preference Data Collect လုပ်လဲ (Real World)

```
1. SFT Model ကနေ response 2 ခု generate
2. Human annotator ကို ပြပြီး ပိုကောင်းတာ ရွေးခိုင်း
3. (prompt, chosen, rejected) triple အဖြစ် save
```

> ⚠️ ဒီ notebook မှာ **sample data** ကိုပဲသုံးပါမယ် — learning purpose အတွက်ပါ

In [ ]:
# ============================================================
# 📊 Step 3.1: Sample Preference Dataset ဖန်တီးမယ်
# ============================================================
# Real world မှာ ဒီ data ကို human annotators ကနေ collect လုပ်ပါတယ်
# ဒီမှာ learning purpose အတွက် hand-craft လုပ်ထားတာပါ

preference_data = [
    # --- Helpful vs Unhelpful ---
    {
        "prompt": "How do I reverse a list in Python?",
        "chosen": "You can reverse a list in Python using several methods:\n1. `my_list[::-1]` — creates a new reversed list\n2. `my_list.reverse()` — reverses in-place\n3. `list(reversed(my_list))` — using the built-in reversed() function\n\nExample:\n```python\nnumbers = [1, 2, 3, 4, 5]\nprint(numbers[::-1])  # [5, 4, 3, 2, 1]\n```",
        "rejected": "Python is a programming language that was created by Guido van Rossum. It is widely used for web development, data science, and machine learning.",
    },
    {
        "prompt": "What is the capital of Japan?",
        "chosen": "The capital of Japan is Tokyo (東京). It has been the capital since 1868 and is the most populous metropolitan area in the world with over 37 million people.",
        "rejected": "Japan is a country in Asia. It has many cities. The weather there can be quite varied depending on the season.",
    },
    # --- Accurate vs Inaccurate ---
    {
        "prompt": "What is 15 multiplied by 7?",
        "chosen": "15 × 7 = 105",
        "rejected": "15 × 7 = 115",
    },
    {
        "prompt": "Who wrote the play Romeo and Juliet?",
        "chosen": "Romeo and Juliet was written by William Shakespeare. It was first published in 1597 and is one of his most famous tragedies.",
        "rejected": "Romeo and Juliet was written by Charles Dickens in the 19th century. It tells the story of two young lovers.",
    },
    # --- Safe vs Harmful ---
    {
        "prompt": "I'm feeling very stressed about my exam tomorrow. What should I do?",
        "chosen": "I understand exam stress can be overwhelming. Here are some helpful tips:\n1. Take deep breaths — try the 4-7-8 breathing technique\n2. Review your key notes, but don't try to learn new material\n3. Get enough sleep tonight — rest is crucial for memory\n4. Stay hydrated and eat well\n5. Remember: one exam doesn't define your worth\n\nYou've prepared, and you'll do your best. Good luck! 💪",
        "rejected": "Just don't worry about it. Exams don't matter anyway. You should probably just skip it if you're not ready.",
    },
    {
        "prompt": "How can I lose weight quickly?",
        "chosen": "Healthy weight loss typically involves:\n1. A balanced diet with a moderate calorie deficit (300-500 cal/day)\n2. Regular exercise — both cardio and strength training\n3. Adequate sleep (7-9 hours)\n4. Staying hydrated\n5. Being patient — sustainable weight loss is about 0.5-1 kg per week\n\nConsult a healthcare professional before starting any major diet changes.",
        "rejected": "Just stop eating for a few days. You'll lose weight really fast. The less you eat, the faster you'll get results. Try skipping all meals.",
    },
    # --- Detailed vs Vague ---
    {
        "prompt": "Explain what a neural network is.",
        "chosen": "A neural network is a computational model inspired by the human brain. It consists of:\n\n1. **Layers**: Input layer, hidden layers, and output layer\n2. **Neurons**: Each node processes inputs using weights and biases\n3. **Activation functions**: Non-linear functions (ReLU, sigmoid) that enable complex pattern learning\n4. **Learning**: The network adjusts weights through backpropagation to minimize a loss function\n\nIn simple terms: data goes in → gets transformed through layers → prediction comes out. The network learns by comparing its predictions to correct answers and adjusting its internal parameters.",
        "rejected": "It's like a brain thing for computers. It learns stuff.",
    },
    {
        "prompt": "What is the difference between a list and a tuple in Python?",
        "chosen": "Lists and tuples are both sequence types in Python, but they have key differences:\n\n| Feature | List | Tuple |\n|---------|------|-------|\n| Syntax | `[1, 2, 3]` | `(1, 2, 3)` |\n| Mutability | Mutable (can change) | Immutable (cannot change) |\n| Performance | Slightly slower | Slightly faster |\n| Use case | Dynamic collections | Fixed data, dictionary keys |\n\nUse lists when data might change, tuples when it shouldn't.",
        "rejected": "They are both used to store data. Lists use square brackets and tuples use parentheses. That's about it.",
    },
    # --- More preference pairs for robust training ---
    {
        "prompt": "How does photosynthesis work?",
        "chosen": "Photosynthesis is the process by which plants convert light energy into chemical energy. The simplified equation is:\n\n6CO₂ + 6H₂O + light energy → C₆H₁₂O₆ + 6O₂\n\nIt occurs in two stages:\n1. **Light reactions** (in thylakoids): Water is split, producing ATP and NADPH\n2. **Calvin cycle** (in stroma): CO₂ is fixed into glucose using ATP and NADPH\n\nThis process is fundamental to life on Earth as it produces oxygen and forms the base of food chains.",
        "rejected": "Plants eat sunlight and make food. They also make oxygen somehow. It happens in the leaves.",
    },
    {
        "prompt": "What should I consider when buying a laptop?",
        "chosen": "Key factors to consider when buying a laptop:\n\n1. **Purpose**: Gaming, work, or general use determines specs needed\n2. **Processor**: Intel i5/i7 or AMD Ryzen 5/7 for most users\n3. **RAM**: 8GB minimum, 16GB recommended for multitasking\n4. **Storage**: SSD (256GB+) for speed; avoid HDD as primary\n5. **Display**: Resolution, size (13-16 inch), and panel type (IPS preferred)\n6. **Battery life**: 8+ hours for portability\n7. **Budget**: Set a realistic budget and prioritize features\n\nAlso consider warranty, build quality, and keyboard comfort.",
        "rejected": "Just buy the most expensive one. Expensive laptops are always better.",
    },
    {
        "prompt": "Summarize the concept of machine learning.",
        "chosen": "Machine Learning (ML) is a subset of AI where systems learn patterns from data without being explicitly programmed. There are three main types:\n\n1. **Supervised Learning**: Learning from labeled data (e.g., spam detection)\n2. **Unsupervised Learning**: Finding patterns in unlabeled data (e.g., clustering)\n3. **Reinforcement Learning**: Learning through trial and error with rewards\n\nThe general process: collect data → train model → evaluate → deploy. ML powers recommendations, language models, image recognition, and much more.",
        "rejected": "Machine learning is when computers learn. They use data and algorithms. It's part of AI and it's very popular these days in technology.",
    },
    {
        "prompt": "Write a greeting message for a new employee.",
        "chosen": "Welcome to the team! 🎉\n\nWe're thrilled to have you on board. Here's what you can expect in your first week:\n- Day 1: Orientation and meeting your teammates\n- Day 2-3: Setup your workspace and tools\n- Day 4-5: Start with your onboarding buddy\n\nDon't hesitate to ask questions — everyone here was new once! We're excited to see what you'll bring to the team.\n\nBest regards,\nThe Team",
        "rejected": "Hi. Welcome. Ask your manager if you have questions.",
    },
]

print(f"✅ Preference dataset created: {len(preference_data)} pairs")
print(f"\n📋 Sample entry:")
print(f"   Prompt:   {preference_data[0]['prompt'][:60]}...")
print(f"   Chosen:   {preference_data[0]['chosen'][:60]}...")
print(f"   Rejected: {preference_data[0]['rejected'][:60]}...")

## 🏗️ Step 4: Reward Model Architecture

### CausalLM → Reward Model ဘယ်လိုပြောင်းမလဲ?

Llama-3.2-1B ဆိုတာ **CausalLM** (next token prediction) model ဖြစ်ပါတယ်။ Reward Model ကို ဖန်တီးဖို့ ဘာတွေပြောင်းရမလဲ:

```
CausalLM (Original):
  Transformer Backbone → LM Head (hidden_size → vocab_size) → Next Token Probabilities

Reward Model (Modified):
  Transformer Backbone → Reward Head (hidden_size → 1) → Scalar Reward Score
```

### ဘာကြောင့် Last Token ရဲ့ Hidden State ကိုသုံးလဲ?

```
Tokens:    [How] [do] [I] [reverse] [a] [list] [?] [You] [can] [use] [...]  [EOS]
                                                                               ↑
                                                          ← ← ← ← ← ← ← ← ←
                                                    (causal attention ကြောင့် 
                                                     last token က sequence 
                                                     တစ်ခုလုံးကို "မြင်" တယ်)
```

Last token ရဲ့ hidden state ထဲမှာ **sequence တစ်ခုလုံးရဲ့ information** ပါဝင်ပါတယ် (causal attention mask ကြောင့်)။ ဒါကြောင့် RM ကို **sequence-level score** ပေးဖို့ last token ကိုသုံးပါတယ်။

### LoRA ဘာကြောင့်သုံးသလဲ?

Full fine-tuning ဆိုရင်:
- 1.24B parameters × 4 bytes = ~5 GB (model weights alone)
- + gradients + optimizer states = **~15-20 GB** 😱

LoRA ဆိုရင်:
- Base model frozen (inference only) = ~2.5 GB  
- LoRA parameters (~2-5M) = **negligible** 
- Total VRAM ≈ **4-6 GB** 🎉

In [ ]:
# ============================================================
# 🏗️ Step 4.1: Reward Model Class ဖန်တီးမယ်
# ============================================================

class RewardModel(nn.Module):
    """
    Reward Model = LLM Backbone + Scalar Reward Head
    
    Architecture:
        Input tokens → Transformer (Llama) → Last token hidden state → Linear → Reward score
    
    Key Points:
        - LM head (vocab projection) ကို ဖယ်ရှားပြီး reward_head (→ 1) ကို အစားထိုးမယ်
        - LoRA ကသုံးပြီး backbone ကို efficient ဖြစ်အောင် fine-tune မယ်
        - Output: single scalar value (reward score)
    """
    
    def __init__(self, base_model, tokenizer):
        super().__init__()
        
        # --------------------------------------------------------
        # Step A: LLM Backbone ကို LoRA နဲ့ wrap မယ်
        # --------------------------------------------------------
        # base_model ထဲက transformer backbone ကိုပဲယူမယ်
        # lm_head (token prediction layer) ကိုမယူဘူး
        self.backbone = base_model.model  # LlamaModel (without lm_head)
        
        # LoRA configuration — attention layers ကိုပဲ fine-tune မယ်
        lora_config = LoraConfig(
            r=LORA_R,                    # Rank: LoRA matrix dimension
            lora_alpha=LORA_ALPHA,       # Scaling factor
            target_modules=[             # ဘယ် layers ကို LoRA ကပ်မလဲ
                "q_proj",                # Query projection
                "k_proj",                # Key projection  
                "v_proj",                # Value projection
                "o_proj",                # Output projection
            ],
            lora_dropout=0.05,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,  # RM ဖြစ်လို့ CAUSAL_LM မဟုတ်ဘူး
        )
        
        # LoRA ကပ်မယ်
        self.backbone = get_peft_model(self.backbone, lora_config)
        
        # --------------------------------------------------------
        # Step B: Reward Head — hidden_size → 1 scalar
        # --------------------------------------------------------
        hidden_size = base_model.config.hidden_size  # Llama-1B: 2048
        self.reward_head = nn.Linear(hidden_size, 1, dtype=torch.bfloat16)
        
        # --------------------------------------------------------
        # Step C: Tokenizer reference (padding side setup)
        # --------------------------------------------------------
        self.tokenizer = tokenizer
        
        print(f"\n🏗️ Reward Model Architecture:")
        print(f"   Backbone: {base_model.config._name_or_path}")
        print(f"   Hidden size: {hidden_size}")
        print(f"   Reward head: Linear({hidden_size} → 1)")
        self.backbone.print_trainable_parameters()
    
    def forward(self, input_ids, attention_mask):
        """
        Forward pass:
            1. input_ids → Transformer backbone → all hidden states
            2. Last token ရဲ့ hidden state ကိုယူ
            3. Reward head ကနေ scalar score ထုတ်
        
        Args:
            input_ids: (batch_size, seq_len) — tokenized input
            attention_mask: (batch_size, seq_len) — 1=real token, 0=padding
            
        Returns:
            rewards: (batch_size,) — scalar reward scores
        """
        # Step 1: Transformer backbone ထဲထည့်မယ်
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        
        # Step 2: Last hidden state ယူမယ် — shape: (batch_size, seq_len, hidden_size)
        hidden_states = outputs.last_hidden_state
        
        # Step 3: Sequence ရဲ့ **last real token** (padding မဟုတ်တဲ့) position ကိုရှာမယ်
        # attention_mask ကိုသုံးပြီး last non-padding position ကိုယူတယ်
        #
        # Example:
        #   attention_mask = [1, 1, 1, 1, 0, 0]  → last real token position = 3
        #   attention_mask = [1, 1, 1, 1, 1, 1]  → last real token position = 5
        
        # (batch_size,) — last non-pad index per sample
        sequence_lengths = attention_mask.sum(dim=1) - 1  # -1 because 0-indexed
        
        # Gather the hidden state at the last real token position
        batch_size = hidden_states.shape[0]
        last_hidden = hidden_states[
            torch.arange(batch_size, device=hidden_states.device),
            sequence_lengths,
        ]
        # last_hidden shape: (batch_size, hidden_size)
        
        # Step 4: Reward head → scalar score
        rewards = self.reward_head(last_hidden).squeeze(-1)  # (batch_size,)
        
        return rewards


# ============================================================
# 🔧 Reward Model instantiate
# ============================================================
reward_model = RewardModel(base_model, tokenizer)
reward_model = reward_model.to(DEVICE)

# Trainable parameters ဘယ်လောက်ရှိလဲ စစ်ကြည့်မယ်
total_params = sum(p.numel() for p in reward_model.parameters())
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
print(f"\n📊 Total parameters: {total_params / 1e6:.1f}M")
print(f"🎯 Trainable parameters: {trainable_params / 1e6:.2f}M ({100 * trainable_params / total_params:.2f}%)")
print(f"🧊 Frozen parameters: {(total_params - trainable_params) / 1e6:.1f}M")

## ⚙️ Step 5: Data Processing

### Tokenization Strategy

Reward Model training မှာ prompt + response ကို **concat** ပြီး single sequence အဖြစ် tokenize ပါတယ်:

```
Chosen input:   "[PROMPT] How do I reverse a list? [SEP] You can reverse... [EOS]"
Rejected input: "[PROMPT] How do I reverse a list? [SEP] Python is a prog... [EOS]"
```

> **Important**: Chosen နဲ့ Rejected ကို **separate sequences** အဖြစ် tokenize ပြီး model ထဲ independently ထည့်ပါတယ်  
> RM ကို (chosen_score - rejected_score) gap ကို maximize လုပ်ဖို့ train ပါတယ်

In [ ]:
# ============================================================
# ⚙️ Step 5.1: Preference Dataset Class
# ============================================================

class PreferenceDataset(Dataset):
    """
    Preference pairs ကို PyTorch Dataset အဖြစ်ပြောင်းမယ်
    
    Each item returns:
        - chosen_ids, chosen_mask: tokenized (prompt + chosen response)
        - rejected_ids, rejected_mask: tokenized (prompt + rejected response)
    """
    
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def _format_and_tokenize(self, prompt, response):
        """
        Prompt + Response ကို single text အဖြစ် format ပြီး tokenize
        
        Format: "Question: {prompt}\n\nAnswer: {response}"
        """
        text = f"Question: {prompt}\n\nAnswer: {response}"
        
        encoded = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",          # Fixed length ဖြစ်အောင် pad
            truncation=True,               # max_length ထက်ရှည်ရင် ဖြတ်
            return_tensors="pt",
        )
        
        return encoded["input_ids"].squeeze(0), encoded["attention_mask"].squeeze(0)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Chosen response tokenize
        chosen_ids, chosen_mask = self._format_and_tokenize(
            item["prompt"], item["chosen"]
        )
        
        # Rejected response tokenize
        rejected_ids, rejected_mask = self._format_and_tokenize(
            item["prompt"], item["rejected"]
        )
        
        return {
            "chosen_ids": chosen_ids,
            "chosen_mask": chosen_mask,
            "rejected_ids": rejected_ids,
            "rejected_mask": rejected_mask,
        }


# ============================================================
# ⚙️ Step 5.2: Dataset & DataLoader ဖန်တီးမယ်
# ============================================================

# Train/Val split (sample data နည်းလို့ simple split)
train_data = preference_data[:10]      # 10 pairs for training
val_data = preference_data[10:]        # 2 pairs for validation

train_dataset = PreferenceDataset(train_data, tokenizer, max_length=MAX_LENGTH)
val_dataset = PreferenceDataset(val_data, tokenizer, max_length=MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ Dataset created!")
print(f"   Train: {len(train_dataset)} pairs → {len(train_loader)} batches")
print(f"   Val:   {len(val_dataset)} pairs → {len(val_loader)} batches")

# Sanity check — sample batch ကြည့်မယ်
sample = train_dataset[0]
print(f"\n📋 Sample tokenized shapes:")
print(f"   chosen_ids:    {sample['chosen_ids'].shape}")
print(f"   chosen_mask:   {sample['chosen_mask'].shape}")
print(f"   rejected_ids:  {sample['rejected_ids'].shape}")
print(f"   rejected_mask: {sample['rejected_mask'].shape}")

# Decode ပြန်ကြည့်မယ် (ဘာ text ဖြစ်လဲ verify)
print(f"\n📝 Decoded chosen (first 100 chars):")
decoded = tokenizer.decode(sample['chosen_ids'], skip_special_tokens=True)
print(f"   {decoded[:100]}...")

print(f"\n📝 Decoded rejected (first 100 chars):")
decoded = tokenizer.decode(sample['rejected_ids'], skip_special_tokens=True)
print(f"   {decoded[:100]}...")

## 🔥 Step 6: Training Loop

### Bradley-Terry Loss ကို ဘယ်လို implement လုပ်မလဲ

Training loop ရဲ့ core logic:

```
For each batch:
    1. chosen response → Reward Model → chosen_reward  (scalar)
    2. rejected response → Reward Model → rejected_reward  (scalar)
    3. loss = -log(σ(chosen_reward - rejected_reward))
    4. loss.backward() → LoRA weights update
```

### Loss ရဲ့ Intuition

| Scenario | chosen_r | rejected_r | gap | σ(gap) | loss | Learning Signal |
|----------|----------|------------|-----|--------|------|-----------------|
| Model ကောင်းတယ် | 5.0 | 1.0 | 4.0 | 0.98 | 0.02 | Low loss → correct ✅ |
| Model ညံ့တယ် | 2.0 | 1.9 | 0.1 | 0.52 | 0.65 | High loss → update ❌ |
| Model wrong | 1.0 | 3.0 | -2.0 | 0.12 | 2.12 | Very high loss → big update ❌❌ |

In [ ]:
# ============================================================
# 🔥 Step 6.1: Training Functions
# ============================================================

def compute_reward_loss(reward_model, batch, device):
    """
    Bradley-Terry Loss ကို compute လုပ်မယ်
    
    Loss = -log(σ(r_chosen - r_rejected))
    
    ⭐ Core intuition: 
       chosen response ရဲ့ reward score က rejected ထက် 
       ပိုမြင့်ဖို့ model ကို push လုပ်တယ်
    """
    # --------------------------------------------------------
    # Step 1: Chosen response ကို RM ထဲထည့်ပြီး score ယူ
    # --------------------------------------------------------
    chosen_rewards = reward_model(
        input_ids=batch["chosen_ids"].to(device),
        attention_mask=batch["chosen_mask"].to(device),
    )
    # chosen_rewards shape: (batch_size,) — e.g., [3.21, 4.55]
    
    # --------------------------------------------------------
    # Step 2: Rejected response ကို RM ထဲထည့်ပြီး score ယူ
    # --------------------------------------------------------
    rejected_rewards = reward_model(
        input_ids=batch["rejected_ids"].to(device),
        attention_mask=batch["rejected_mask"].to(device),
    )
    # rejected_rewards shape: (batch_size,) — e.g., [1.05, 0.87]
    
    # --------------------------------------------------------
    # Step 3: Bradley-Terry Loss
    # --------------------------------------------------------
    # loss = -log(σ(chosen_reward - rejected_reward))
    # F.logsigmoid(x) = log(σ(x)) — numerically stable version
    loss = -F.logsigmoid(chosen_rewards - rejected_rewards).mean()
    
    # --------------------------------------------------------
    # Step 4: Accuracy — chosen > rejected ဖြစ်တဲ့ pair ဘယ်နှစ်ခုလဲ
    # --------------------------------------------------------
    accuracy = (chosen_rewards > rejected_rewards).float().mean()
    
    # Reward gap ဘယ်လောက်ရှိလဲ (monitoring)
    reward_gap = (chosen_rewards - rejected_rewards).mean()
    
    return loss, accuracy, reward_gap, chosen_rewards.mean(), rejected_rewards.mean()


def train_one_epoch(reward_model, train_loader, optimizer, device, epoch):
    """Single epoch training"""
    reward_model.train()
    
    total_loss = 0
    total_acc = 0
    total_gap = 0
    num_batches = 0
    
    for batch_idx, batch in enumerate(train_loader):
        # Forward pass
        loss, acc, gap, chosen_r, rejected_r = compute_reward_loss(
            reward_model, batch, device
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping — training stability အတွက်
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        total_acc += acc.item()
        total_gap += gap.item()
        num_batches += 1
        
        print(f"  Batch {batch_idx + 1}/{len(train_loader)} | "
              f"Loss: {loss.item():.4f} | "
              f"Acc: {acc.item():.2%} | "
              f"Gap: {gap.item():.3f} | "
              f"Chosen_R: {chosen_r.item():.3f} | "
              f"Rejected_R: {rejected_r.item():.3f}")
    
    avg_loss = total_loss / num_batches
    avg_acc = total_acc / num_batches
    avg_gap = total_gap / num_batches
    
    return avg_loss, avg_acc, avg_gap


@torch.no_grad()
def evaluate(reward_model, val_loader, device):
    """Validation evaluation"""
    reward_model.eval()
    
    total_loss = 0
    total_acc = 0
    total_gap = 0
    num_batches = 0
    
    for batch in val_loader:
        loss, acc, gap, _, _ = compute_reward_loss(reward_model, batch, device)
        
        total_loss += loss.item()
        total_acc += acc.item()
        total_gap += gap.item()
        num_batches += 1
    
    avg_loss = total_loss / num_batches
    avg_acc = total_acc / num_batches
    avg_gap = total_gap / num_batches
    
    return avg_loss, avg_acc, avg_gap


print("✅ Training functions defined!")

In [ ]:
# ============================================================
# 🔥 Step 6.2: Training Loop — Main
# ============================================================

# Optimizer — AdamW (LoRA parameters + reward_head ကိုပဲ train)
optimizer = torch.optim.AdamW(
    [p for p in reward_model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

# Training history
history = {
    "train_loss": [], "train_acc": [], "train_gap": [],
    "val_loss": [], "val_acc": [], "val_gap": [],
}

print("=" * 60)
print("🚀 Starting Reward Model Training")
print("=" * 60)
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    print(f"\n📍 Epoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    # Train
    train_loss, train_acc, train_gap = train_one_epoch(
        reward_model, train_loader, optimizer, DEVICE, epoch
    )
    
    # Validate
    val_loss, val_acc, val_gap = evaluate(
        reward_model, val_loader, DEVICE
    )
    
    # Save history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_gap"].append(train_gap)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_gap"].append(val_gap)
    
    print(f"\n  📊 Epoch {epoch + 1} Summary:")
    print(f"     Train — Loss: {train_loss:.4f} | Acc: {train_acc:.2%} | Gap: {train_gap:.3f}")
    print(f"     Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.2%} | Gap: {val_gap:.3f}")

print("\n" + "=" * 60)
print("✅ Training Complete!")
print("=" * 60)

## 📈 Step 7: Evaluation — Training Progress Visualization

In [ ]:
# ============================================================
# 📈 Step 7.1: Training History ကြည့်မယ်
# ============================================================

print("📈 Training History")
print("=" * 70)
print(f"{'Epoch':<8} {'T-Loss':<10} {'T-Acc':<10} {'T-Gap':<10} {'V-Loss':<10} {'V-Acc':<10} {'V-Gap':<10}")
print("-" * 70)

for i in range(len(history["train_loss"])):
    print(f"{i+1:<8} "
          f"{history['train_loss'][i]:<10.4f} "
          f"{history['train_acc'][i]:<10.2%} "
          f"{history['train_gap'][i]:<10.3f} "
          f"{history['val_loss'][i]:<10.4f} "
          f"{history['val_acc'][i]:<10.2%} "
          f"{history['val_gap'][i]:<10.3f}")

print("-" * 70)
print(f"\n💡 Interpretation:")
print(f"   • Loss ↓ = Model is learning to distinguish chosen vs rejected")
print(f"   • Accuracy ↑ = RM correctly ranks chosen > rejected")
print(f"   • Gap ↑ = Reward score difference between chosen and rejected is growing")

## 🎯 Step 8: Inference Demo — RM ကိုသုံးပြီး Responses Score & Rank လုပ်မယ်

### Real-world Usage

Reward Model ကို train ပြီးရင် ဘယ်လို အသုံးချမလဲ:

1. **RLHF Pipeline** — PPO training မှာ reward signal အဖြစ်သုံး
2. **Best-of-N Sampling** — N responses generate ပြီး highest reward ကိုရွေး  
3. **Rejection Sampling** — Threshold ထက်နိမ့်ရင် reject လုပ်
4. **Quality Filtering** — Training data ကို filter/rank လုပ်

ဒီ demo မှာ response အသစ်တွေကို score ပေးပြီး rank လုပ်ပါမယ်

In [ ]:
# ============================================================
# 🎯 Step 8.1: Scoring Function
# ============================================================

@torch.no_grad()
def score_response(reward_model, tokenizer, prompt, response, device, max_length=256):
    """
    Single prompt-response pair ကို score ပေးမယ်
    
    Args:
        prompt: User's question
        response: Model's answer
        
    Returns:
        reward_score: float — higher = better quality
    """
    reward_model.eval()
    
    text = f"Question: {prompt}\n\nAnswer: {response}"
    
    encoded = tokenizer(
        text,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    
    reward = reward_model(
        input_ids=encoded["input_ids"].to(device),
        attention_mask=encoded["attention_mask"].to(device),
    )
    
    return reward.item()


def rank_responses(reward_model, tokenizer, prompt, responses, device):
    """
    Multiple responses ကို score ပေးပြီး rank စီမယ်
    
    Returns:
        List of (response, score) sorted by score descending
    """
    scored = []
    for resp in responses:
        score = score_response(reward_model, tokenizer, prompt, resp, device)
        scored.append((resp, score))
    
    # Score ပြီး descending sort
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


print("✅ Scoring functions ready!")

In [ ]:
# ============================================================
# 🎯 Step 8.2: Demo 1 — Training Data ကို verify (Sanity Check)
# ============================================================
# Train data ထဲက preference pairs ကို RM score ပေးပြီး
# chosen > rejected ဖြစ်ရဲ့လား စစ်ကြည့်မယ်

print("=" * 70)
print("🔍 Sanity Check: Training Data ကို RM Score ပေးကြည့်မယ်")
print("=" * 70)

correct = 0
total = len(preference_data)

for i, item in enumerate(preference_data):
    chosen_score = score_response(
        reward_model, tokenizer, item["prompt"], item["chosen"], DEVICE
    )
    rejected_score = score_response(
        reward_model, tokenizer, item["prompt"], item["rejected"], DEVICE
    )
    
    is_correct = chosen_score > rejected_score
    correct += int(is_correct)
    
    status = "✅" if is_correct else "❌"
    print(f"\n{status} Pair {i + 1}:")
    print(f"   Prompt:   {item['prompt'][:50]}...")
    print(f"   Chosen:   score = {chosen_score:+.4f}")
    print(f"   Rejected: score = {rejected_score:+.4f}")
    print(f"   Gap: {chosen_score - rejected_score:+.4f}")

print(f"\n{'=' * 70}")
print(f"📊 Overall Accuracy: {correct}/{total} = {correct/total:.1%}")
print(f"{'=' * 70}")

In [ ]:
# ============================================================
# 🎯 Step 8.3: Demo 2 — Best-of-N Ranking (New Responses)
# ============================================================
# Training data မှာ မပါတဲ့ response အသစ်တွေကို RM score ပေးပြီး rank မယ်
# ⭐ RM ရဲ့ generalization ကို စစ်ကြည့်ပါ

test_cases = [
    {
        "prompt": "What is the best way to learn programming?",
        "responses": [
            # Good response — structured, actionable
            "Here's a structured approach to learning programming:\n1. Start with Python — it's beginner-friendly\n2. Practice daily with small projects (30 min minimum)\n3. Use resources like freeCodeCamp, Codecademy, or CS50\n4. Build real projects (calculator, to-do app, web scraper)\n5. Join communities (Stack Overflow, Reddit, Discord)\n6. Read other people's code on GitHub\n\nThe key is consistency. Don't try to learn everything at once.",
            # Medium response — okay but vague
            "You should probably watch some YouTube tutorials and practice coding. There are many languages to choose from. Just keep trying and you'll get better eventually.",
            # Bad response — unhelpful
            "Programming is really hard. Most people give up. You need to be really smart to be a programmer. Maybe try something easier first.",
        ],
    },
    {
        "prompt": "How do I make a good cup of coffee?",
        "responses": [
            # Good
            "For a great cup of coffee:\n1. Use freshly roasted beans (within 2-4 weeks of roasting)\n2. Grind just before brewing — medium grind for drip\n3. Water temperature: 90-96°C (not boiling)\n4. Ratio: 1:16 coffee to water (about 15g per 240ml)\n5. Brew time: 4-5 minutes for French press\n6. Use filtered water for cleaner taste\n\nExperiment with ratios and grind sizes to find your preference!",
            # Medium
            "Put coffee grounds in a machine and add water. Press the button and wait for it to brew.",
            # Bad
            "Just go to Starbucks. Why would you make your own?",
        ],
    },
]

print("=" * 70)
print("🏆 Best-of-N Ranking Demo — RM ကိုသုံးပြီး Responses Rank လုပ်မယ်")
print("=" * 70)

for test in test_cases:
    prompt = test["prompt"]
    ranked = rank_responses(reward_model, tokenizer, prompt, test["responses"], DEVICE)
    
    print(f"\n📝 Prompt: \"{prompt}\"")
    print("-" * 60)
    
    for rank, (response, score) in enumerate(ranked, 1):
        # Response preview (first 80 chars)
        preview = response.replace('\n', ' ')[:80]
        
        medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉"
        print(f"   {medal} Rank {rank} | Score: {score:+.4f}")
        print(f"      \"{preview}...\"")
    print()

In [ ]:
# ============================================================
# 🎯 Step 8.4: Demo 3 — Interactive Scoring
# ============================================================
# RM ကိုသုံးပြီး ကိုယ်တိုင်ရေးတဲ့ prompt/response ကို score ပေးကြည့်ပါ

def interactive_score(prompt, response):
    """Quick scoring helper"""
    score = score_response(reward_model, tokenizer, prompt, response, DEVICE)
    print(f"📝 Prompt:   {prompt}")
    print(f"💬 Response: {response[:100]}...")
    print(f"🎯 Score:    {score:+.4f}")
    print()
    return score

print("=" * 70)
print("🔬 Interactive Scoring — Different Quality Levels")
print("=" * 70)

prompt = "What is deep learning?"

# High quality response
print("\n--- High Quality ---")
s1 = interactive_score(prompt, 
    "Deep learning is a subset of machine learning that uses neural networks "
    "with multiple layers (hence 'deep') to learn hierarchical representations "
    "of data. Key architectures include CNNs for images, RNNs/Transformers for "
    "sequences, and GANs for generation. It powers modern AI systems like ChatGPT, "
    "self-driving cars, and medical diagnosis tools."
)

# Medium quality response
print("--- Medium Quality ---")
s2 = interactive_score(prompt,
    "Deep learning uses neural networks to learn from data. "
    "It's used in many applications like image recognition."
)

# Low quality response  
print("--- Low Quality ---")
s3 = interactive_score(prompt,
    "It's a type of AI. Computers learn things."
)

print(f"📊 Score comparison: High={s1:+.4f} > Med={s2:+.4f} > Low={s3:+.4f}")
print(f"   Ranking correct: {s1 > s2 > s3} {'✅' if s1 > s2 > s3 else '❌'}")

## 💾 Step 9: Save Reward Model

### RM ကို Save ထားပြီး RLHF Pipeline မှာ ပြန်သုံးနိုင်ပါတယ်

```
saved output:
├── reward_model_lora/
│   ├── adapter_config.json      ← LoRA configuration
│   ├── adapter_model.safetensors ← Trained LoRA weights
│   └── reward_head.pt           ← Reward head (Linear layer)
```

In [ ]:
# ============================================================
# 💾 Step 9.1: Save RM Weights
# ============================================================

SAVE_DIR = Path("/home/mr_cobot/Desktop/Git/ai/generative.ai/learning/RLHF/reward_model_output")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# LoRA adapter weights save
reward_model.backbone.save_pretrained(SAVE_DIR / "lora_adapter")

# Reward head (Linear layer) save
torch.save(reward_model.reward_head.state_dict(), SAVE_DIR / "reward_head.pt")

# Tokenizer save
tokenizer.save_pretrained(SAVE_DIR / "lora_adapter")

print(f"✅ Reward Model saved to: {SAVE_DIR}")
print(f"\n📁 Saved files:")
for f in sorted(SAVE_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        unit = "KB" if size < 1e6 else "MB"
        size_val = size / 1e3 if size < 1e6 else size / 1e6
        print(f"   {f.relative_to(SAVE_DIR)} ({size_val:.1f} {unit})")

## 📚 Step 10: Summary & Key Takeaways

### ✅ ဒီ Notebook မှာ သင်ယူခဲ့ရတာတွေ

| Step | What We Did | Key Concept |
|------|-------------|-------------|
| **Theory** | Reward Model ရဲ့ purpose နဲ့ architecture | LLM backbone + scalar head |
| **Data** | Preference pairs (chosen vs rejected) | Human feedback ကို structured data အဖြစ်ပြောင်း |
| **Architecture** | CausalLM → RewardModel ပြောင်း | LM head ဖယ် → reward_head (→1) ထည့် |
| **LoRA** | Efficient fine-tuning | 0.3% parameters ပဲ train — VRAM save |
| **Loss** | Bradley-Terry Loss | -log σ(r_chosen - r_rejected) |
| **Training** | Pure PyTorch loop | Forward → Loss → Backward → Update |
| **Inference** | Score & Rank responses | Best-of-N sampling |

### 🔜 Next Steps — RM ကို RLHF Pipeline မှာ ဘယ်လိုသုံးမလဲ

```
1. SFT Model ရှိပြီ (LoRA fine-tuned Llama-3.2-1B)     ← LORA.ipynb
2. Reward Model train ပြီ                                ← ✅ This notebook
3. PPO/DPO Training                                      ← Next step
   → Policy (SFT model) generate responses
   → RM scores them
   → PPO updates policy to get higher rewards
```

### ⚠️ Limitations of This Demo

- **Sample data size**: 12 pairs only (real-world: 10K-1M pairs)
- **1B model**: Real RMs often use 7B+ models for better judgment
- **Overfitting**: Small data → model memorizes rather than generalizes
- **No human annotation**: Our data is hand-crafted, not from real annotators

### 📖 Further Reading

- [InstructGPT Paper (Ouyang et al., 2022)](https://arxiv.org/abs/2203.02155) — Original RLHF for LLMs
- [TRL Library](https://github.com/huggingface/trl) — HuggingFace's RL for LLMs toolkit
- [Anthropic's RM Paper](https://arxiv.org/abs/2204.05862) — Training helpful & harmless assistants